In [ ]:
import pandas as pd
import os
from datetime import datetime

def get_space_weather_indicators(target_date_str, data_dir='omni_data_monthly'):
    """
    Pobiera wskaźniki pogody kosmicznej z lokalnych plików NASA OMNI 
    dla podanej daty i zwraca je w formacie zgodnym z Twoimi JSON-ami.
    """
    # 1. Konwersja wejściowej daty na obiekt datetime
    try:
        target_dt = pd.to_datetime(target_date_str)
    except Exception as e:
        return f"Błąd formatu daty: {e}"

    # 2. Wyznaczenie nazwy pliku na podstawie roku i miesiąca
    file_label = target_dt.strftime('%Y_%m')
    file_path = os.path.join(data_dir, f'omni_hourly_{file_label}.csv')

    if not os.path.exists(file_path):
        return f"Brak danych dla daty {file_label} (plik {file_path} nie istnieje)."

    # 3. Wczytanie danych z odpowiedniego miesiąca
    df = pd.read_csv(file_path)
    df['Time'] = pd.to_datetime(df['Time'])

    # 4. Znalezienie najbliższego wpisu (OMNI ma dane co godzinę, np. o 14:30)
    # Metoda 'nearest' znajdzie godzinę najbliższą Twojemu lotowi
    df = df.sort_values('Time')
    idx = df['Time'].searchsorted(target_dt)
    
    # Wybieramy najbliższy wiersz
    if idx == 0:
        closest_row = df.iloc[0]
    elif idx == len(df):
        closest_row = df.iloc[-1]
    else:
        # Sprawdzamy, czy poprzedni czy następny wiersz jest bliżej
        before = df.iloc[idx-1]
        after = df.iloc[idx]
        if abs(after['Time'] - target_dt) < abs(before['Time'] - target_dt):
            closest_row = after
        else:
            closest_row = before

    # 5. MAPOWANIE: NASA OMNI -> Twoje JSON-y
    # Tutaj definiujemy, która kolumna z NASA odpowiada której w Twoim modelu
    mapping = {
        'KP1800': 'Index_Kp',
        'DST1800': 'Index_Dst',
        'AP_INDEX1800': 'Index_Ap',
        'R1800': 'Solar_sunspots',
        'F10_INDEX1800': 'Solar_f107',
        'ABS_B1800': 'SW_B',
        'BZ_GSM1800': 'SW_Bz',
        'V1800': 'SW_V',
        'N1800': 'SW_density',
        'T1800': 'SW_temperature',
        'Pressure1800': 'SW_pressure',
        'PR-FLX_101800': 'Particles_P10',
        'PR-FLX_301800': 'Particles_P30',
        'PR-FLX_601800': 'Particles_P60'
    }

    # Budujemy wynikowy słownik
    result = {
        "Requested_Datetime": str(target_dt),
        "Found_OMNI_Time": str(closest_row['Time'])
    }

    for nasa_name, json_name in mapping.items():
        if nasa_name in closest_row:
            result[json_name] = closest_row[nasa_name]

    return result

# --- PRZYKŁAD UŻYCIA ---
# data_lotu = "2023-03-23 23:54:16"
# wskazniki = get_space_weather_indicators(data_lotu)
# print(wskazniki)

In [1]:
from hapiclient import hapi
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

def get_live_space_weather(target_date_str):
    """
    Łączy się bezpośrednio z API NASA, pobiera dane dla konkretnej daty
    i mapuje je na format Twoich JSON-ów.
    """
    server     = 'https://cdaweb.gsfc.nasa.gov/hapi'
    dataset    = 'OMNI2_H0_MRG1HR'
    
    # 1. Przygotowanie okna czasowego (pobieramy 1 godzinę wokół daty)
    target_dt = pd.to_datetime(target_date_str)
    start_str = (target_dt - timedelta(minutes=30)).strftime('%Y-%m-%dT%H:%M:%SZ')
    stop_str  = (target_dt + timedelta(minutes=30)).strftime('%Y-%m-%dT%H:%M:%SZ')

    # Pełna lista parametrów (tak jak wcześniej ustaliliśmy)
    all_params = [
        'ABS_B1800', 'BZ_GSM1800', 'V1800', 'N1800', 'T1800', 'Pressure1800',
        'R1800', 'F10_INDEX1800', 'KP1800', 'DST1800', 'AP_INDEX1800',
        'PR-FLX_101800', 'PR-FLX_301800', 'PR-FLX_601800'
    ]
    params_str = ",".join(all_params)

    try:
        print(f"Odpytywanie NASA API dla daty: {target_dt}...")
        data, meta = hapi(server, dataset, params_str, start_str, stop_str)
        
        if len(data) == 0:
            return "Brak danych w NASA dla tego zakresu."

        # Tworzymy tymczasowy DataFrame dla jednego wiersza
        df = pd.DataFrame(data)
        
        # --- INTELIGENTNE CZYSZCZENIE (Fill Values) ---
        for p in meta['parameters']:
            col = p['name']
            fill_val = p.get('fill')
            if fill_val is not None:
                fv = float(fill_val)
                df[col] = df[col].replace(fv, np.nan)
                # Obsługa precyzji float
                if fv > 0: df.loc[df[col] >= fv * 0.999, col] = np.nan

        # Wybieramy pierwszy (najbliższy) wiersz
        row = df.iloc[0]

        # --- MAPOWANIE NA TWOJE NAZWY ---
        mapping = {
            'KP1800': 'Index_Kp',
            'DST1800': 'Index_Dst',
            'AP_INDEX1800': 'Index_Ap',
            'R1800': 'Solar_sunspots',
            'F10_INDEX1800': 'Solar_f107',
            'ABS_B1800': 'SW_B',
            'BZ_GSM1800': 'SW_Bz',
            'V1800': 'SW_V',
            'N1800': 'SW_density',
            'T1800': 'SW_temperature',
            'Pressure1800': 'SW_pressure',
            'PR-FLX_101800': 'Particles_P10',
            'PR-FLX_301800': 'Particles_P30',
            'PR-FLX_601800': 'Particles_P60'
        }

        result = {"Datetime": target_date_str}
        for nasa, target in mapping.items():
            result[target] = row[nasa]

        return result

    except Exception as e:
        return f"Błąd połączenia z API: {e}"

# --- TEST ---
# wynik = get_live_space_weather("2023-03-23 23:54:16")
# print(wynik)

ModuleNotFoundError: No module named 'hapiclient'